In [ ]:
! pip install duckdb

In [10]:
import duckdb

try:
    con.close() 
except NameError:
    pass

con = duckdb.connect("../dataset/goodreads.db") 
con.sql("SET memory_limit='4GB'")
con.sql("SET temp_directory='duckdb_tmp'")

In [3]:
con.sql("""
    CREATE OR REPLACE TABLE user_stats AS
    SELECT user_id,
           COUNT(*) AS n_interactions,
           SUM(CASE WHEN rating > 0 THEN 1 ELSE 0 END) AS n_rated
    FROM interactions
    GROUP BY user_id
""")

In [4]:
con.sql("""
    CREATE OR REPLACE TABLE book_stats AS
    SELECT book_id,
           COUNT(*) AS n_interactions,
           SUM(CASE WHEN rating > 0 THEN 1 ELSE 0 END) AS n_rated
    FROM interactions
    GROUP BY book_id
""")

In [5]:
con.sql("""
    SELECT 'users' AS what,
           COUNT(*) AS total,
           MEDIAN(n_rated) AS median_rated,
           quantile_cont(n_rated, 0.9) AS p90_rated,
           MAX(n_rated) AS max_rated,
           SUM(CASE WHEN n_rated >= 20 THEN 1 ELSE 0 END) AS with_20_plus_rated
    FROM user_stats
    UNION ALL
    SELECT 'books',
           COUNT(*),
           MEDIAN(n_rated),
           quantile_cont(n_rated, 0.9),
           MAX(n_rated),
           SUM(CASE WHEN n_rated >= 20 THEN 1 ELSE 0 END)
    FROM book_stats
""").show()

┌─────────┬─────────┬──────────────┬───────────┬───────────┬────────────────────┐
│  what   │  total  │ median_rated │ p90_rated │ max_rated │ with_20_plus_rated │
│ varchar │  int64  │    double    │  double   │  int128   │       int128       │
├─────────┼─────────┼──────────────┼───────────┼───────────┼────────────────────┤
│ users   │  876145 │         49.0 │     295.0 │     38884 │             639453 │
│ books   │ 2360643 │          4.0 │      42.0 │    281135 │             434531 │
└─────────┴─────────┴──────────────┴───────────┴───────────┴────────────────────┘



In [8]:
#use iterative k core filteration
con.sql("""
    CREATE OR REPLACE TABLE kcore AS
    SELECT user_id, book_id, rating, is_read, is_reviewed
    FROM interactions
    WHERE rating > 0
""")


for i in range(1, 11):
    before = con.sql("SELECT COUNT(*) FROM kcore").fetchone()[0]

    con.sql("""
        CREATE OR REPLACE TABLE kcore_next AS
        SELECT * FROM kcore
        WHERE user_id IN (SELECT user_id FROM kcore GROUP BY user_id HAVING COUNT(*) >= 20)
          AND book_id IN (SELECT book_id FROM kcore GROUP BY book_id HAVING COUNT(*) >= 20)
    """)
    con.sql("DROP TABLE kcore")
    con.sql("ALTER TABLE kcore_next RENAME TO kcore")

    after = con.sql("SELECT COUNT(*) FROM kcore").fetchone()[0]
    print(f"Round {i}: {before:,} -> {after:,} rows")
    if before == after:
        print("Stable, done.")
        break



con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM kcore
""").show()

Round 1: 104,551,537 -> 94,313,824 rows
Round 2: 94,313,824 -> 94,045,805 rows
Round 3: 94,045,805 -> 94,001,311 rows
Round 4: 94,001,311 -> 93,997,057 rows
Round 5: 93,997,057 -> 93,996,052 rows
Round 6: 93,996,052 -> 93,995,938 rows
Round 7: 93,995,938 -> 93,995,919 rows
Round 8: 93,995,919 -> 93,995,919 rows
Stable, done.
┌──────────┬─────────┬─────────┐
│  n_rows  │ n_users │ n_books │
│  int64   │  int64  │  int64  │
├──────────┼─────────┼─────────┤
│ 93995919 │  626362 │  428655 │
└──────────┴─────────┴─────────┘



In [11]:
#sampling by users
#randomly select 10k users (pseudorandom) should give about 1.5m rows
con.sql("""
    CREATE OR REPLACE TABLE sample_users AS
    SELECT DISTINCT user_id
    FROM kcore
    ORDER BY hash(user_id)
    LIMIT 10000
""")

con.sql("""
    CREATE OR REPLACE TABLE sample_data AS
    SELECT * FROM kcore
    WHERE user_id IN (SELECT user_id FROM sample_users)
""")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM sample_data
""").show()

┌─────────┬─────────┬─────────┐
│ n_rows  │ n_users │ n_books │
│  int64  │  int64  │  int64  │
├─────────┼─────────┼─────────┤
│ 1503365 │   10000 │  254864 │
└─────────┴─────────┴─────────┘



In [12]:
#filtration again
#first select a cutoff
con.sql("""
    WITH book_count AS (
        SELECT book_id, COUNT(*) AS n
        FROM sample_data
        GROUP BY book_id
    )
    SELECT t AS min_ratings_per_book,
           COUNT(*) FILTER (WHERE n >= t) AS books_kept,
           SUM(n) FILTER (WHERE n >= t) AS rows_kept
    FROM book_count, (VALUES (1), (3), (5), (10), (20)) AS cutoffs(t)
    GROUP BY t
    ORDER BY t
""").show()

┌──────────────────────┬────────────┬───────────┐
│ min_ratings_per_book │ books_kept │ rows_kept │
│        int32         │   int64    │  int128   │
├──────────────────────┼────────────┼───────────┤
│                    1 │     254864 │   1503365 │
│                    3 │      88360 │   1284324 │
│                    5 │      48927 │   1151902 │
│                   10 │      22644 │    983471 │
│                   20 │      10404 │    820376 │
└──────────────────────┴────────────┴───────────┘



In [13]:
#selected cutoff is 10
con.sql("""
    CREATE OR REPLACE TABLE sample_final AS
    SELECT * FROM sample_data
    WHERE book_id IN (
        SELECT book_id FROM sample_data
        GROUP BY book_id HAVING COUNT(*) >= 10
    )
""")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM sample_final
""").show()

con.sql("""
    SELECT MIN(n) AS min_per_user, MEDIAN(n) AS median_per_user
    FROM (SELECT user_id, COUNT(*) AS n FROM sample_final GROUP BY user_id)
""").show()

┌────────┬─────────┬─────────┐
│ n_rows │ n_users │ n_books │
│ int64  │  int64  │  int64  │
├────────┼─────────┼─────────┤
│ 983471 │    9993 │   22644 │
└────────┴─────────┴─────────┘

┌──────────────┬─────────────────┐
│ min_per_user │ median_per_user │
│    int64     │     double      │
├──────────────┼─────────────────┤
│            1 │            55.0 │
└──────────────┴─────────────────┘



In [14]:
#the previous filtration step left at least one user with a single rating, which we should remove...we need at least 20 ratings per person
#do another kcore filtration
con.sql("""
    CREATE OR REPLACE TABLE final_core AS
    SELECT * FROM sample_final
""")

for i in range(1, 11):
    before = con.sql("SELECT COUNT(*) FROM final_core").fetchone()[0]

    con.sql("""
        CREATE OR REPLACE TABLE final_next AS
        SELECT * FROM final_core
        WHERE user_id IN (SELECT user_id FROM final_core GROUP BY user_id HAVING COUNT(*) >= 20)
          AND book_id IN (SELECT book_id FROM final_core GROUP BY book_id HAVING COUNT(*) >= 10)
    """)
    con.sql("DROP TABLE final_core")
    con.sql("ALTER TABLE final_next RENAME TO final_core")

    after = con.sql("SELECT COUNT(*) FROM final_core").fetchone()[0]
    print(f"Round {i}: {before:,} -> {after:,} rows")
    if before == after:
        print("Stable, done.")
        break

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT user_id) AS n_users,
           COUNT(DISTINCT book_id) AS n_books
    FROM final_core
""").show()

con.sql("""
    SELECT MIN(n) AS min_per_user, MEDIAN(n) AS median_per_user
    FROM (SELECT user_id, COUNT(*) AS n FROM final_core GROUP BY user_id)
""").show()

Round 1: 983,471 -> 968,813 rows
Round 2: 968,813 -> 964,346 rows
Round 3: 964,346 -> 963,561 rows
Round 4: 963,561 -> 963,189 rows
Round 5: 963,189 -> 963,020 rows
Round 6: 963,020 -> 962,922 rows
Round 7: 962,922 -> 962,922 rows
Stable, done.
┌────────┬─────────┬─────────┐
│ n_rows │ n_users │ n_books │
│ int64  │  int64  │  int64  │
├────────┼─────────┼─────────┤
│ 962922 │    8825 │   22070 │
└────────┴─────────┴─────────┘

┌──────────────┬─────────────────┐
│ min_per_user │ median_per_user │
│    int64     │     double      │
├──────────────┼─────────────────┤
│           20 │            65.0 │
└──────────────┴─────────────────┘



In [15]:
con.close()